# Day 1 — Whisper Base baseline
Goal: transcribe a few clips with `openai/whisper-base`, record raw predictions, runtime and WER/CER.
Language prompting is a **controlled variable**: compare auto / hindi / english rather than assuming one helps.

In [ ]:
# --- Setup (run once per Colab session) ---
REPO_URL = "https://github.com/arkalatua/Indic_Switch.git"
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Indic_Switch"):
        subprocess.run(["git", "clone", REPO_URL, "/content/Indic_Switch"], check=True)
    %cd /content/Indic_Switch
    !pip install -q -r requirements.txt
    # Optional: persist models/results across sessions
    # from google.colab import drive; drive.mount("/content/drive")
else:
    os.chdir(os.path.abspath(".."))  # running locally from notebooks/
sys.path.insert(0, os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Audio fundamentals: waveform, log-Mel spectrogram, duration

In [ ]:
import numpy as np, matplotlib.pyplot as plt, librosa, librosa.display
from src.preprocessing import load_audio

AUDIO = "data/samples/sample.wav"   # upload your own clip (or record one in the next cell)
wav, sr = load_audio(AUDIO)
print(f"{len(wav)/sr:.2f}s @ {sr} Hz, mono, peak amplitude {np.abs(wav).max():.3f}")
mel = librosa.power_to_db(librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=80), ref=np.max)
fig, ax = plt.subplots(2, 1, figsize=(12, 5))
ax[0].plot(np.arange(len(wav))/sr, wav); ax[0].set_title("waveform")
librosa.display.specshow(mel, sr=sr, x_axis="time", y_axis="mel", ax=ax[1]); ax[1].set_title("log-Mel (80 bins, as Whisper uses)")
plt.tight_layout()

## Transcribe with three language settings

In [ ]:
from src.asr import DecodeConfig, WhisperASR
from src.evaluation import corpus_wer

REFERENCE = ""  # optional: type the true transcript to get WER/CER
results = {}
for lang in [None, "hindi", "english"]:
    asr = WhisperASR(DecodeConfig(language=lang, num_beams=3))
    r = asr.transcribe(AUDIO)
    results[lang or "auto"] = r
    print(f"[{lang or 'auto':7s}] RTF {r.rtf:.3f} | {r.text}")
    if REFERENCE:
        s = corpus_wer([REFERENCE], [r.text]); a = corpus_wer([REFERENCE], [r.text], romanize=True)
        print(f"          WER {s['wer']:.3f} CER {s['cer']:.3f} | script-agnostic WER {a['wer']:.3f}")

## Baseline over a manifest (E1 / E2)
After Day 2 produces a manifest, run the baseline on the held-out split. Each run caches predictions to `experiments/asr/`.

In [ ]:
MANIFEST = "data/manifests/coshe500.csv"   # built in notebook 02
!python scripts/run_asr.py --manifest {MANIFEST} --split test                       # E1: default decoding
!python scripts/run_asr.py --manifest {MANIFEST} --split test --language hindi --beams 3   # E2
!python scripts/run_asr.py --manifest {MANIFEST} --split test --language english --beams 3

In [ ]:
import json, glob, shutil
rows = []
for f in sorted(glob.glob("experiments/asr/*.summary.json")):
    s = json.load(open(f))
    rows.append({"config": s["tag"], "n": s["n"], "WER": round(s["scores"]["wer"], 4), "CER": round(s["scores"]["cer"], 4),
                 "WER_script_agnostic": round(s["scores_script_agnostic"]["wer"], 4), "RTF": round(s["rtf_excl_model_load"], 3),
                 "GPU": s.get("device"), "peak_GB": s.get("peak_gpu_mem_gb")})
import pandas as pd; df = pd.DataFrame(rows); display(df)
json.dump(rows, open("experiments/baseline.json", "w"), indent=2)